# IMDB Sentiment Classification — Colab Training Notebook
Use this notebook to do all GPU-heavy work (Model V1 + V2 training, MLflow tracking) on Google Colab's free GPU.

**Workflow:** Clone your GitHub repo here -> train -> push trained `artifacts/` folder back to GitHub (or download it) -> use it locally on Windows for Docker/Kubernetes/API steps.

Steps:
1. Set Runtime -> Change runtime type -> GPU (T4)
2. Run the cells below in order

In [ ]:
# 1. Clone your repo (replace with your GitHub URL after you've pushed the project)
!git clone https://github.com/YOUR_USERNAME/YOUR_REPO.git
%cd YOUR_REPO

In [ ]:
# 2. Install dependencies
!pip install -q -r requirements.txt

In [ ]:
# 3. Confirm GPU is available
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 4. Train Model V1 (LSTM baseline)
!python training/train_v1.py

In [ ]:
# 5. Train Model V2 (fine-tuned DistilBERT) -- benefits most from GPU
!python training/train_v2.py

In [ ]:
# 6. Compare both models (writes artifacts/model_comparison_report.csv)
!python training/compare_models.py

In [ ]:
# 7. (Optional) View MLflow UI inside Colab using pyngrok
!pip install -q pyngrok
import subprocess, time
from pyngrok import ngrok

# Get a free token at https://dashboard.ngrok.com/get-started/your-authtoken
# ngrok.set_auth_token('YOUR_NGROK_TOKEN')

get_ipython().system_raw('mlflow ui --port 5000 &')
time.sleep(5)
public_url = ngrok.connect(5000)
print('MLflow UI:', public_url)

In [ ]:
# 8. Zip artifacts + mlruns to download locally for use on Windows
!zip -r trained_artifacts.zip artifacts mlruns
from google.colab import files
files.download('trained_artifacts.zip')

In [ ]:
# 9. (Optional) Commit and push trained artifacts back to GitHub
# Note: large model weights should normally use Git LFS or be excluded via .gitignore
# and re-downloaded/retrained instead of committed directly.
!git config --global user.email "you@example.com"
!git config --global user.name "Your Name"
# !git add artifacts/model_comparison_report.csv
# !git commit -m "Add training results from Colab"
# !git push